In [1]:
import os
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

In [2]:
cornwall_granular_collection = Chroma(
    collection_name="cornwall_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)

In [3]:
cornwall_granular_collection.reset_collection()

In [4]:
cornwall_coarse_collection = Chroma(
    collection_name="cornwall_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)
cornwall_granular_collection.reset_collection()

In [5]:
os.environ["USER_AGENT"] = "manning-ch08/1.0 (mic.a.elle.chlon@gmail.com)"

from langchain_community.document_loaders import AsyncHtmlLoader
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()

Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  5.02it/s]


In [6]:
from langchain_text_splitters import HTMLSectionSplitter

header_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=header_to_split_on
)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(
            html_string
        )
        all_chunks.extend(temp_chunks)

    return all_chunks

In [7]:
granular_chunks = split_docs_into_granular_chunks(docs)

cornwall_granular_collection.add_documents(documents=granular_chunks)

['4c5be052-ce91-46d4-b8d4-8bb42703702a',
 'eeaf1596-f17e-4e9a-85a7-089c6b43abea',
 'c420c52e-81bd-4756-8c4f-3df68e32b9c3',
 '2e234bf9-a2b6-4857-89f6-ff1aec21f17b',
 'f4e7269c-ad86-4379-b17d-21b8b90181f1',
 'c72013b2-e7f4-4773-88c3-654a267a9317',
 'a6f15b16-0845-4447-aab4-27d7cbe5d81a',
 '284f14c7-f906-43ec-bbdf-73d534b549e1',
 '3835289d-8158-4317-a2ce-fe73c7d9e7a0',
 '6f564f04-2c81-4580-a514-27e9af91bea5',
 'dd8d50a5-904a-4040-ba11-033a7e303955',
 'ba13d890-a4b6-4f53-9fdd-1688435853fe',
 'aadf125d-1962-4e6a-b214-5c38ae5b3b3a',
 '538fe5d4-523e-42b3-9e16-e7dc1a121637',
 'fcf2c042-c2a8-4742-8ea2-affdbbc0a3ef',
 '99735046-c460-42f2-9396-3447e78f3697',
 '3ffb225c-86b6-466b-9589-fa0db0c573c8',
 '80ddec9d-b91c-4c54-89a4-14314b5ff568',
 'ee57f540-4255-4149-8123-3311347ea585']

In [8]:
# results = cornwall_granular_collection.similarity_search(
#     query="Events or festivals in Cornwall",k=3
# )

# for doc in results:
#     print(doc)

In [9]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [10]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)
    coarse_chunks = text_splitter.split_documents(text_docs)
    return coarse_chunks

In [11]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

cornwall_coarse_collection.add_documents(documents=coarse_chunks)

['2efcf71d-601b-40fb-bfc0-6cd63ecc85ff',
 '478a212c-7cca-457f-806c-9c1f5bd12693',
 'a37b02fc-ea78-4e85-8419-0a6e28d91e05',
 'c942e920-ff0a-4868-a410-aaff6a91f293',
 '7ff16069-273e-4f50-80cf-27b0ec8838fd',
 '4bc4868e-f64d-47ad-869c-bc75e46f0247',
 'c5674018-619e-4201-b190-5a124e53196e',
 '4064a0be-0cc1-4de1-b1af-c5b67d31879d',
 '65f5640f-f8f1-4785-9787-4356e5aa7895',
 'e9c5ebd5-c1f6-48ab-b66f-edff56b78109',
 '7bbe93b4-effd-4107-bd23-a829a8091f10',
 '0476734f-4466-4c81-8859-05a6935645db',
 '9f9057b8-dd92-4bd2-8a03-5c7f18da877d',
 '826a1751-7ad0-4a7c-a54d-34e5cc37cad4',
 '2eb66a03-aecd-4736-bebb-cb6b572d4bf9']

In [12]:
# results = cornwall_coarse_collection.similarity_search(
#     query="Events or Festival in Cornwall", k=3
# )

# for doc in results:
#     print(doc)

In [13]:
uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_granular_collection.reset_collection()

uk_coarse_collection = Chroma(
    collection_name="uk_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_coarse_collection.reset_collection()

uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' 
                       for d in uk_destinations]

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    docs = html_loader.load()

granular_chunks = split_docs_into_granular_chunks(docs)
uk_granular_collection.add_documents(documents=granular_chunks)

coarse_chunks = split_docs_into_coarse_chunks(docs)
uk_coarse_collection.add_documents(documents=coarse_chunks)

Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  7.08it/s]


['0ed06bc6-dd46-443c-bb95-df9dc1a6f3c5',
 '4541db3a-b047-4aaf-9c90-3f25c5ce3511',
 'b3d5bd26-0d49-427a-b16e-b0d6ef2a94c8',
 'af40c7e4-a857-42d1-b826-3c6dc0e35b4d',
 'cf1dd67a-298c-42fa-96f2-44cf2b1b10f3',
 'f1cad21c-5c1c-4892-bdeb-0a5103007a78',
 '4765c694-c541-47d4-85ba-4f668374fb10',
 '347f6031-81a0-47f3-a303-5137cfde8a40',
 'f86eba78-45b1-4fc9-9fd0-b11630fa0972']

In [14]:
# granular_results = uk_granular_collection.similarity_search(
#     query="Events or festivals in East Sussex",k=4)

# for doc in granular_results:
#     print(doc)
#     print("\n------------------------------------------------------------------\n")

# coarse_results = uk_coarse_collection.similarity_search(
#     query="Events or festivals in East Sussex",k=4)

# for doc in coarse_results:
#     print(doc)

## 8.4/ Embedding strategy

In [15]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
# parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
# child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

# child_chunk_collection = Chroma(
#     collection_name="uk_child_chunks",
#     embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY) 
# )
# child_chunk_collection.reset_collection()

# doc_store = InMemoryStore()

# parent_doc_retriever = ParentDocumentRetriever(
#     vectorstore=child_chunk_collection,
#     docstore=doc_store,
#     child_splitter=child_splitter,
#     parent_splitter=parent_splitter
# )

In [17]:
# for destination_url in uk_destination_urls:
#     html_loader = AsyncHtmlLoader(destination_url)
#     html_docs = html_loader.load()
#     text_docs = html2text_transformer.transform_documents(html_docs)
#     print(f'Ingesting {destination_url}')
#     parent_doc_retriever.add_documents(text_docs, ids=None)

In [18]:
# list(doc_store.yield_keys())

In [19]:
# retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [20]:
# print(retrieved_docs[0])

In [21]:
# child_docs_only = child_chunk_collection.similarity_search("Cornwall Ranger")

In [22]:
# print(child_docs_only)

### 8.4.2/ Embedding child chunks with MultiVectorRetriever

In [23]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [24]:
# parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
# child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

# child_chunks_collection = Chroma(
#     collection_name="uk_child_chunks",
#     embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY) 
# )
# child_chunks_collection.reset_collection()

# doc_byte_store = InMemoryByteStore()
# doc_key = "doc_id"

# multi_vector_retriever = MultiVectorRetriever(
#     vectorstore=child_chunks_collection,
#     byte_store=doc_byte_store
# )

In [25]:
# for destination_url in uk_destination_urls:
#     html_loader = AsyncHtmlLoader(destination_url)
#     html_docs = html_loader.load()
#     text_docs = html2text_transformer.transform_documents(html_docs)

#     coarse_chunks = parent_splitter.split_documents(text_docs)
    
#     coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    
#     all_granular_chunks = []
#     for i, coarse_chunk in enumerate(coarse_chunks):

#         coarse_chunk_id = coarse_chunks_ids[i]
#         granular_chunks = child_splitter.split_documents([coarse_chunk])

#         for granular_chunk in granular_chunks:
#             granular_chunk.metadata[doc_key] = coarse_chunk_id

#         all_granular_chunks.extend(granular_chunks)
    
#     print(f'Ingesting {destination_url}')
#     multi_vector_retriever.vectorstore.add_documents(all_granular_chunks)
#     multi_vector_retriever.docstore.mset(
#         list(zip(coarse_chunks_ids, coarse_chunks)))

In [26]:
# retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
# print(retrieved_docs)

In [27]:
# child_docs_only = child_chunks_collection.similarity_search("Cornwall Ranger")
# print(child_docs_only[0])

### 8.4.3/ Embedding document summaries 

In [28]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

In [29]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)

summaries_collection = Chroma(
    collection_name="uk_summaries",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)
summaries_collection.reset_collection()

In [30]:
doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

In [31]:
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)

In [32]:
llm = ChatOpenAI(model="gpt-5-nano", api_key=OPENAI_API_KEY)

summarization_chain = (
    {"document": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}")
    | llm
    | StrOutputParser()
)

In [33]:
# for destination_url in uk_destination_urls:
#     html_loader = AsyncHtmlLoader(destination_url)
#     html_docs = html_loader.load()

#     text_docs = html2text_transformer.transform_documents(html_docs)

#     coarse_chunks = parent_splitter.split_documents(text_docs)
#     coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
#     all_summaries = []
#     for i, coarse_chunk in enumerate(coarse_chunks):
#         coarse_chunk_id = coarse_chunks_ids[i]
#         summary_text = summarization_chain.invoke(coarse_chunk)
#         summary_doc = Document(page_content=summary_text,
#                                metadata={doc_key: coarse_chunk_id})
#         all_summaries.append(summary_doc)
#     print(f'Ingesting {destination_url}')
#     multi_vector_retriever.vectorstore.add_documents(all_summaries)
#     multi_vector_retriever.docstore.mset(
#         list(zip(coarse_chunks_ids, coarse_chunks)))

In [34]:
# retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

# summary_docs_only = summaries_collection.similarity_search("Cornwall Travel")
# print(summary_docs_only[0])

### 8.4.4/ Embedding hypothetical questions

In [35]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

In [36]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)

hypothetical_questions_collection = Chroma(
    collection_name="uk_hypothetical_questions",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
hypothetical_questions_collection.reset_collection()

doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)

In [37]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""
    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

llm_with_structured_output = ChatOpenAI(
    model="gpt-5-nano",
    api_key=OPENAI_API_KEY).with_structured_output(HypotheticalQuestions)

In [38]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Generate a list of exactly 4 hypothetical questions asking to generate that the below text could be used to answer: four hypothetical\n\n{document_text}"
    )
    | llm_with_structured_output
    | (lambda x: x.questions)
)

In [39]:
# for destination_url in uk_destination_urls:
#     html_loader = AsyncHtmlLoader(destination_url)
#     html_docs = html_loader.load()
#     text_docs = html2text_transformer.transform_documents(html_docs)

#     coarse_chunks = parent_splitter.split_documents(text_docs)
#     coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]

#     all_hypothetical_questions = []
#     for i, coarse_chunk in enumerate(coarse_chunks):
#         coarse_chunk_id = coarse_chunks_ids[i]
#         hypothetical_questions = hypothetical_questions_chain.invoke(coarse_chunk)
#         hypothetical_questions_docs = [Document(
#             page_content=question, metadata={doc_key: coarse_chunk_id})
#                                        for question in hypothetical_questions]
#         all_hypothetical_questions.extend(hypothetical_questions_docs)

#     print(f'Ingesting {destination_url}')
#     multi_vector_retriever.vectorstore.add_documents(all_hypothetical_questions)
#     multi_vector_retriever.docstore.mset(
#         list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  5.03it/s]
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...ional Trust gardens)?"]), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que... mining heritage?", '']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_

Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  5.47it/s]
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...rth Cornwall article?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...atch Cornish hurling?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_

Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  5.53it/s]
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...about South Cornwall?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...s largest greenhouse?"]), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_

...

Ingesting https://en.wikivoyage.org/wiki/Porthleven


Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  6.00it/s]
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...the East Sussex page?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...in the provided text?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_

Ingesting https://en.wikivoyage.org/wiki/East_Sussex


Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  4.43it/s]
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...bus, train, and taxi?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/home/mic/DEV/AI_AGENTS_APPS_Manning/ch08/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que... granted city status?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_

KeyboardInterrupt: 

In [ ]:
# retrieved_docs = multi_vector_retriever.invoke("How can you go to Brighton from London?")

# print(retrieved_docs[0])

In [40]:
# hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search("How can you go to Brighton from London?")

## 8.5/ Granular chunk expression